In [1]:
# Import libraries used for dataset inspection and creation.

import pandas as pd
import ast

In [2]:
# Load the original Amazon Electronics dataset.
# The raw file will never be modified.

raw_path = "../raw/Amazon Electronics Metadata.csv"

df = pd.read_csv(raw_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

Dataset loaded successfully.
Shape: (498196, 9)


In [3]:
# Display the basic structure of the raw dataset.

print("Columns:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

Columns:
['asin', 'imUrl', 'description', 'categories', 'title', 'price', 'salesRank', 'related', 'brand']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 498196 entries, 0 to 498195
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   asin         498196 non-null  object 
 1   imUrl        498021 non-null  object 
 2   description  442139 non-null  object 
 3   categories   498196 non-null  object 
 4   title        491192 non-null  object 
 5   price        389693 non-null  float64
 6   salesRank    128706 non-null  object 
 7   related      366959 non-null  object 
 8   brand        141365 non-null  object 
dtypes: float64(1), object(8)
memory usage: 34.2+ MB


In [4]:
# Check missing values in each column.

missing_values = df.isnull().sum()

print("Missing values:")
print(missing_values)

print("\nMissing percentage:")
missing_percentage = (df.isnull().sum() / len(df)) * 100
print(missing_percentage.round(2))

Missing values:
asin                0
imUrl             175
description     56057
categories          0
title            7004
price          108503
salesRank      369490
related        131237
brand          356831
dtype: int64

Missing percentage:
asin            0.00
imUrl           0.04
description    11.25
categories      0.00
title           1.41
price          21.78
salesRank      74.17
related        26.34
brand          71.62
dtype: float64


In [5]:
# Check for duplicate rows and duplicate product IDs (ASINs).

print("Duplicate rows:", df.duplicated().sum())
print("Duplicate ASINs:", df["asin"].duplicated().sum())

Duplicate rows: 0
Duplicate ASINs: 0


In [6]:
# Check the datatype of every column.

print(df.dtypes)

asin            object
imUrl           object
description     object
categories      object
title           object
price          float64
salesRank       object
related         object
brand           object
dtype: object


In [ ]:
# Inspect the price 

print(df["price"].describe())

print("\nNegative prices:", (df["price"] < 0).sum())
print("Zero prices:", (df["price"] == 0).sum())

count    389693.000000
mean         61.406786
std         119.118870
min           0.010000
25%           9.950000
50%          19.990000
75%          51.950000
max         999.990000
Name: price, dtype: float64

Negative prices: 0
Zero prices: 0


In [8]:
# Inspect the fields that will be used by the recommendation system.

print("Sample titles:")
display(df["title"].dropna().head(5))

print("\nSample descriptions:")
display(df["description"].dropna().head(3))

print("\nSample categories:")
display(df["categories"].head(5))

print("\nSample related products:")
display(df["related"].dropna().head(3))

Sample titles:


0    Kelby Training DVD: Mastering Blend Modes in A...
1    Kelby Training DVD: Adobe Photoshop CS5 Crash ...
2                      Digital Organizer and Messenger
3    CLIKR-5 Time Warner Cable Remote Control UR5U-...
4    Rand McNally 528881469 7-inch Intelliroute TND...
Name: title, dtype: object


Sample descriptions:


0    The Kelby Training DVD Mastering Blend Modes i...
2                      Digital Organizer and Messenger
3    The CLIKR-5 UR5U-8780L remote control is desig...
Name: description, dtype: object


Sample categories:


0    [['Electronics', 'Computers & Accessories', 'C...
1    [['Electronics', 'Computers & Accessories', 'C...
2    [['Electronics', 'Computers & Accessories', 'P...
3    [['Electronics', 'Accessories & Supplies', 'Au...
4    [['Electronics', 'GPS & Navigation', 'Vehicle ...
Name: categories, dtype: object


Sample related products:


2    {'also_viewed': ['0545016266', 'B009ECM8QY', '...
3    {'also_viewed': ['B001KC08A4', 'B00KUL8O0W', '...
4    {'also_viewed': ['B006ZOI9OY', 'B00C7FKT2A', '...
Name: related, dtype: object

In [9]:
# Identify products that have neither a title nor a description.
# These products cannot provide useful textual information
# for our recommendation system.

missing_text = (
    df["title"].isna()
    & df["description"].isna()
)

print(
    "Products missing both title and description:",
    missing_text.sum()
)

Products missing both title and description: 1734


In [34]:
# Select only the fields required by our recommendation system.

required_columns = [
    "asin",
    "title",
    "description",
    "categories",
    "price",
    "brand",
    "imUrl",
    "related"
]

products_df = df[required_columns].copy()

print("Selected fields:")
print(products_df.columns.tolist())
print("\nShape:", products_df.shape)

Selected fields:
['asin', 'title', 'description', 'categories', 'price', 'brand', 'imUrl', 'related']

Shape: (498196, 8)


In [36]:
# Remove products that have neither a title nor a description.
# Products with at least one text field are retained.

missing_text = (
    products_df["title"].isna()
    & products_df["description"].isna()
)

recommendation_df = products_df.loc[~missing_text].copy()

print("Original products:", len(products_df))
print("Removed products:", missing_text.sum())
print("Recommendation candidates:", len(recommendation_df))

Original products: 498196
Removed products: 1734
Recommendation candidates: 496462


In [37]:
# Convert category strings into Python lists so that
# we can work with the category hierarchy.

recommendation_df["category_list"] = (
    recommendation_df["categories"].apply(ast.literal_eval)
)

In [38]:
# Extract the final category from each category hierarchy.
# This will help us distribute products across different categories.

recommendation_df["main_category"] = (
    recommendation_df["category_list"]
    .apply(
        lambda x: x[0][-1]
        if x and x[0]
        else None
    )
)

In [39]:
# Count how many products are available in each category.

category_counts = recommendation_df["main_category"].value_counts()

print("Number of categories:", len(category_counts))

display(category_counts.head(30))

Number of categories: 781


main_category
Cases                            39023
Batteries                        21104
Chargers & Adapters              16753
Headphones                       11566
Laptops                           9470
Camera Cases                      9054
Sleeves & Slipcases               6993
Memory                            6841
Camera Batteries                  6645
USB Cables                        6360
Screen Protectors                 6325
Skins & Decals                    5969
AC Adapters                       5838
USB Flash Drives                  5400
Speaker Systems                   5230
Keyboards                         4867
HDMI Cables                       4643
Computers & Accessories           4622
Point & Shoot Digital Cameras     4476
Lamps                             4289
Accessory Kits                    4240
MP3 Players                       4025
Mice                              4007
Desktops                          3935
Connectors & Adapters             3825
Monitors   

In [40]:
# Select a representative group of products from each category.
# random_state ensures that the same products are selected
# every time the notebook is run.

core_products = (
    recommendation_df
    .dropna(subset=["main_category"])
    .groupby("main_category", group_keys=False)
    .apply(
        lambda group: group.sample(
            n=min(len(group), 15),
            random_state=42
        )
    )
    .reset_index(drop=True)
)

print("Core products selected:", len(core_products))
print(
    "Categories represented:",
    core_products["main_category"].nunique()
)

Core products selected: 10495
Categories represented: 781


C:\Users\areeb\AppData\Local\Temp\ipykernel_16212\1841360035.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


In [41]:
# Keep the core product set at a maximum of 500 products.

if len(core_products) > 500:
    core_products = (
        core_products
        .sample(
            n=500,
            random_state=42
        )
        .reset_index(drop=True)
        )

print("Core product count:", len(core_products))

Core product count: 500


In [42]:
# Create a set of all available ASINs.
# This lets us quickly check whether a related product
# actually exists in the source dataset.

available_asins = set(
    recommendation_df["asin"]
)

print("Available ASINs:", len(available_asins))

Available ASINs: 496462


In [43]:
# Extract product ASINs from the "related" field.

def extract_related_asins(value):

    if pd.isna(value):
        return []

    try:
        related_data = ast.literal_eval(value)

        if not isinstance(related_data, dict):
            return []

        related_asins = []

        for asin_list in related_data.values():

            if isinstance(asin_list, list):
                related_asins.extend(asin_list)

        return related_asins

    except (ValueError, SyntaxError):
        return []

In [44]:
# Find related products that also exist in our source dataset.

related_asins = set()

for value in core_products["related"]:

    for asin in extract_related_asins(value):

        if asin in available_asins:
            related_asins.add(asin)

print(
    "Related products found:",
    len(related_asins)
)

Related products found: 6328


In [45]:
# Add up to 100 related products.
# This keeps the prototype dataset small while preserving
# useful relationships for alternative recommendations.

related_asins = list(related_asins)

if len(related_asins) > 100:
    related_asins = related_asins[:100]

related_products = recommendation_df[
    recommendation_df["asin"].isin(related_asins)
].copy()

print(
    "Related products selected:",
    len(related_products)
)

Related products selected: 100


In [46]:
# Combine the core products with the selected related products.

final_df = pd.concat(
    [
        core_products,
        related_products
    ],
    ignore_index=True
)

# Remove any duplicate products.

final_df = (
    final_df
    .drop_duplicates(subset=["asin"])
    .reset_index(drop=True)
)

print("Final dataset shape:", final_df.shape)

Final dataset shape: (600, 10)


In [47]:
# Verify that every product has a unique ASIN.

print(
    "Duplicate ASINs:",
    final_df["asin"].duplicated().sum()
)

Duplicate ASINs: 0


In [48]:
# Check availability of important recommendation fields.

print("Products with title:",
      final_df["title"].notna().sum())

print("Products with description:",
      final_df["description"].notna().sum())

print("Products with price:",
      final_df["price"].notna().sum())

print("Products with related data:",
      final_df["related"].notna().sum())

Products with title: 594
Products with description: 529
Products with price: 479
Products with related data: 495


In [49]:
# Check the category distribution in the final dataset.

print(
    "Categories represented:",
    final_df["main_category"].nunique()
)

display(
    final_df["main_category"]
    .value_counts()
    .head(30)
)

Categories represented: 405


main_category
Subwoofers                              7
Film                                    4
Golf Course GPS Units                   4
Amps                                    4
TV Ceiling & Wall Mounts                4
Rear Projection TV Replacement Lamps    4
DVD+RW Discs                            4
Home Theater Systems                    4
Mounts                                  4
Vehicle Electronics Accessories         3
Complete Tripods                        3
Speaker Cables                          3
Laptop Barebones                        3
Fans & Cooling                          3
Archival Storage Binders                3
PDAs, Handhelds & Accessories           3
Polarizing Filters                      3
Overhead Video                          3
Line Cords                              3
Posing Props                            3
Darkroom Supplies                       3
APS Cameras                             3
Connectors & Adapters                   3
In-Dash DVD & Video 

In [ ]:
# Keep only the 8 fields required by the application.

final_df = final_df[
    [
        "asin",
        "title",
        "description",
        "categories",
        "price",
        "related",
        "imUrl",
        "brand"
    ]
].copy()

#Rename the original Amazon image URL column to a clearer
# application-friendly name.

final_df = final_df.rename(
    columns={
        "imUrl": "image_url"
    }
)


print("Final columns:")
print(final_df.columns.tolist())

Final columns:
['asin', 'title', 'description', 'categories', 'price', 'related', 'image_url', 'brand']


In [33]:
print(final_df.columns.tolist())

['asin', 'title', 'description', 'categories', 'price', 'related']


In [51]:
# Final check before creating the CSV file.

print("Final dataset shape:", final_df.shape)

print("\nDuplicate ASINs:",
      final_df["asin"].duplicated().sum())

print("\nMissing values:")
print(final_df.isnull().sum())

print("\nSample products:")
display(final_df.head(10))

Final dataset shape: (600, 8)

Duplicate ASINs: 0

Missing values:
asin             0
title            6
description     71
categories       0
price          121
related        105
image_url        0
brand          370
dtype: int64

Sample products:


,asin,title,description,categories,price,related,image_url,brand
0,B000ICL3AG,Fujifilm xD-Picture Card Type M 2GB,SanDisk 2GB XD Type M Picture Card \n\n,"[['Electronics', 'Computers & Accessories', 'P...",NaN,NaN,http://ecx.images-amazon.com/images/I/51DPpnrR...,NaN
1,B0037QF9UK,Samsung HW-C451 Soundbar with Wireless Sub (Br...,HW-C451,"[['Electronics', 'Home Audio', 'Stereo Compone...",195.61,"{'also_viewed': ['B00BLX9510', 'B00DDTMJOU', '...",http://ecx.images-amazon.com/images/I/31-uuLDb...,NaN
2,B0030CE73G,OmniMount NC200T Black Tilt Mount for 37-63 in...,Tilt -5&#xB0; to +15&#xB0; to reduce glare. Un...,"[['Electronics', 'Portable Audio & Video', 'MP...",129.00,"{'also_viewed': ['B00BCA41RA', 'B00BMGTALQ', '...",http://ecx.images-amazon.com/images/I/41p75sw5...,OmniMount
3,B005ZAFVIQ,EzFoto 49mm Adapter Ring + 49mm Black Pro1 Sup...,49mm Filter Adapter Ring and Pro MCUV Filter f...,"[['Electronics', 'Camera & Photo', 'Accessorie...",9.99,"{'also_viewed': ['B008KFY16Q', 'B0059VMH5G', '...",http://ecx.images-amazon.com/images/I/41d7On%2...,NaN
4,B008XAZHN4,ZyXEL Wireless N 300Mbps Range Extender (WRE2205),Designed as a user-friendly alternative to com...,"[['Electronics', 'Computers & Accessories', 'N...",44.90,"{'also_bought': ['B0061308MA', 'B00BZBZZVW', '...",http://ecx.images-amazon.com/images/I/31Bso0g%...,ZyXel
5,B00IGKU87Q,Wilson Electronics AG Pro Quint Cell Phone Sig...,NaN,"[['Cell Phones & Accessories', 'Accessories', ...",768.99,"{'also_bought': ['B0018PR1H6', 'B0018PXRN8', '...",http://ecx.images-amazon.com/images/I/41tRzf8d...,Wilson Electronics
6,B00DTPYWBG,Huion 4 x 2.23 Inches OSU Tablet Graphics Draw...,Technology\nElectromagnetic DigitizerActive Ar...,"[['Electronics', 'Computers & Accessories', 'C...",24.50,"{'also_bought': ['B008I646WG', 'B00DOW6TUQ', '...",http://ecx.images-amazon.com/images/I/31NNugja...,NaN
7,B001XVCMTM,Universal Power Group D1294 SS15 15 Watt Trump...,15 Watt trumpet Style 2-Tone siren for indoor ...,"[['Electronics', 'Security & Surveillance', 'H...",15.39,"{'also_bought': ['B004W0EQIG'], 'buy_after_vie...",http://ecx.images-amazon.com/images/I/21XOuuiB...,NaN
8,B007T4MDTG,Pink Electric Portable Office desk USB Mini Fa...,Pink Electric Portable Office desk USB Mini Fa...,"[['Electronics', 'Computers & Accessories', 'C...",NaN,"{'also_viewed': ['B00BC4CIIA', 'B00K734EUM', '...",http://ecx.images-amazon.com/images/I/417OLd-G...,NaN
9,B004G8ATE4,Pyle PLVWR1544 15.1-Inch Flip Down Roof Mount ...,"This 15.1"" flip-down Pyle high-resolution TFT ...","[['Electronics', 'Car & Vehicle Electronics', ...",110.45,"{'also_bought': ['B009NVXGAI', 'B0049LBJ56', '...",http://ecx.images-amazon.com/images/I/417hQtpU...,Pyle


In [52]:
# Save the final recommendation dataset.
# The original raw dataset remains untouched.

output_path = "../processed/products.csv"

final_df.to_csv(
    output_path,
    index=False
)

print("Final dataset saved successfully.")
print("Location:", output_path)

Final dataset saved successfully.
Location: ../processed/products.csv


In [53]:
# Read the saved CSV again to make sure it was created correctly.

check_df = pd.read_csv(output_path)

print("Saved dataset shape:", check_df.shape)

print("\nColumns:")
print(check_df.columns.tolist())

print("\nDuplicate ASINs:",
      check_df["asin"].duplicated().sum())

display(check_df.head())

Saved dataset shape: (600, 8)

Columns:
['asin', 'title', 'description', 'categories', 'price', 'related', 'image_url', 'brand']

Duplicate ASINs: 0


,asin,title,description,categories,price,related,image_url,brand
0,B000ICL3AG,Fujifilm xD-Picture Card Type M 2GB,SanDisk 2GB XD Type M Picture Card \n\n,"[['Electronics', 'Computers & Accessories', 'P...",NaN,NaN,http://ecx.images-amazon.com/images/I/51DPpnrR...,NaN
1,B0037QF9UK,Samsung HW-C451 Soundbar with Wireless Sub (Br...,HW-C451,"[['Electronics', 'Home Audio', 'Stereo Compone...",195.61,"{'also_viewed': ['B00BLX9510', 'B00DDTMJOU', '...",http://ecx.images-amazon.com/images/I/31-uuLDb...,NaN
2,B0030CE73G,OmniMount NC200T Black Tilt Mount for 37-63 in...,Tilt -5&#xB0; to +15&#xB0; to reduce glare. Un...,"[['Electronics', 'Portable Audio & Video', 'MP...",129.00,"{'also_viewed': ['B00BCA41RA', 'B00BMGTALQ', '...",http://ecx.images-amazon.com/images/I/41p75sw5...,OmniMount
3,B005ZAFVIQ,EzFoto 49mm Adapter Ring + 49mm Black Pro1 Sup...,49mm Filter Adapter Ring and Pro MCUV Filter f...,"[['Electronics', 'Camera & Photo', 'Accessorie...",9.99,"{'also_viewed': ['B008KFY16Q', 'B0059VMH5G', '...",http://ecx.images-amazon.com/images/I/41d7On%2...,NaN
4,B008XAZHN4,ZyXEL Wireless N 300Mbps Range Extender (WRE2205),Designed as a user-friendly alternative to com...,"[['Electronics', 'Computers & Accessories', 'N...",44.90,"{'also_bought': ['B0061308MA', 'B00BZBZZVW', '...",http://ecx.images-amazon.com/images/I/31Bso0g%...,ZyXel


In [54]:
import pandas as pd

check_df = pd.read_csv("../processed/products.csv")

print("Shape:", check_df.shape)

print("\nColumns:")
print(check_df.columns.tolist())

print("\nDuplicate ASINs:")
print(check_df["asin"].duplicated().sum())

Shape: (600, 8)

Columns:
['asin', 'title', 'description', 'categories', 'price', 'related', 'image_url', 'brand']

Duplicate ASINs:
0
